# InkCalc — Optuna hyperparameter search (Colab)

Rough search over **`learning_rate`** and **`warmup_ratio`** on a *random subset* of
MathWriting-2024, on a free Colab GPU. Dropout is fixed at 0.15. **No model
checkpoints are saved** — the only output is a good hyperparameter pair to reuse
for the full training run on the `main` branch.

**This run's config**
| setting | value |
|---|---|
| batch size | 32 (safe for a T4) |
| subset | 5,000 train / 1,000 val (random, seeded) |
| schedule | 20 epochs × up to 20 trials (MedianPruner kills weak trials early) |
| study | persisted to SQLite on Drive → **resumable** after a disconnect |

Runtime: set **Runtime ▸ Change runtime type ▸ GPU (T4)** before running.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install deps & mount Drive

In [ ]:
!pip -q install optuna
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo (`acunha_training_loop`)

`training_loop.py` uses namespace packages (no `__init__.py`) and expects
`InkCalc/model` on `sys.path`, so we `cd` into it and add it to the path.

In [ ]:
import os, sys, pathlib
REPO_DIR  = pathlib.Path('/content/InkCalc')
if not REPO_DIR.exists():
    !git clone -b acunha_training_loop https://github.com/alanna-jc/InkCalc.git /content/InkCalc
else:
    print('repo present; pulling latest')
    !cd /content/InkCalc && git pull --ff-only

MODEL_DIR = REPO_DIR / 'model'
os.chdir(MODEL_DIR)
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))
print('cwd     :', os.getcwd())
print('on path :', str(MODEL_DIR))

## 4. Configuration

In [ ]:
from pathlib import Path
import random, json, pickle

# ── Data locations (your Drive) ─────────────────────────────────────────────
TGZ_PATH        = Path('/content/drive/MyDrive/mathwriting-2024.tgz')
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/inkcalc')
EXTRACT_DIR     = Path('/content/mathwriting-2024')
FULL_TRAIN = EXTRACT_DIR / 'mathwriting-2024' / 'train'
FULL_VAL   = EXTRACT_DIR / 'mathwriting-2024' / 'valid'

# ── Search config ───────────────────────────────────────────────────────────
SEED       = 42
N_TRAIN    = 5000
N_VAL      = 1000
BATCH_SIZE = 32
NUM_EPOCHS = 20
N_TRIALS   = 20
STUDY_NAME = 'inkcalc_lr_warmup_v1'   # bump this to start a fresh study

# ── Drive cache artifacts (make resume instant + subset reproducible) ────────
DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
STORAGE_URL    = f'sqlite:///{DRIVE_CACHE_DIR / (STUDY_NAME + ".db")}'
VOCAB_CACHE    = DRIVE_CACHE_DIR / f'{STUDY_NAME}_vocab.json'
SUBSET_CACHE   = DRIVE_CACHE_DIR / f'{STUDY_NAME}_subset.json'
FEATURES_CACHE = DRIVE_CACHE_DIR / f'{STUDY_NAME}_features.pkl'
BEST_PARAMS    = DRIVE_CACHE_DIR / f'{STUDY_NAME}_best_params.json'

print('study storage:', STORAGE_URL)
print('feature cache:', FEATURES_CACHE)

## 5. Import the repo's training code

`training_loop.py` already contains the in-memory feature cache (`CachedDataset`,
`precompute_items`) and the refactored `objective(trial, train_loader, val_loader,
vocab_size, device)`. We reuse those directly. `NUM_EPOCHS` is read by
`objective()` as a module global, so we override it on the module.

In [ ]:
import torch
import training_loop as tl
from vocab import build_vocab, save_vocab, BLANK_IDX
from preprocessing.dataset import MathWritingDataset, _collate_fn, MAX_POINTS
from preprocessing.inkml_parser import InkMLParser, InkMLParseError

tl.NUM_EPOCHS = NUM_EPOCHS          # objective() uses this global for the LR schedule
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device      :', device)
print('NUM_EPOCHS  :', tl.NUM_EPOCHS)
print('MAX_POINTS  :', MAX_POINTS)
print('model       :', f'{tl.NUM_LAYERS} layers, embed {tl.EMBED_DIM}, ffn {tl.FFN_NUM_HIDDEN}, dropout {tl.DROPOUT}')

## 6. Build (or load) the subset + precomputed features

**First run:** streams the `.tgz` twice — once to list `.inkml` members, once to
extract only the seeded random subset (so we never unpack all ~230k files) — then
builds a vocab from the train subset and precomputes features. All of it is
cached to Drive.

**Later runs / resume:** loads the feature pickle straight from Drive and skips
the archive entirely. The cached features *are* the subset, so a resumed study
always trains on exactly the same data.

> Vocab here is built from the **train subset** (fast, self-consistent for the
> search). For the real full-training run on `main`, rebuild the vocab from the
> **full** train split.

In [ ]:
def _collect_labels_from_paths(paths, use_normalized=True):
    parser = InkMLParser(require_time=False)
    labels = []
    for p in paths:
        try:
            s = parser.parse(p)
        except (InkMLParseError, FileNotFoundError):
            continue
        meta_lower = {k.lower(): v for k, v in s.metadata.items()}
        if use_normalized and 'normalizedlabel' in meta_lower:
            labels.append(meta_lower['normalizedlabel'])
        elif s.label:
            labels.append(s.label)
    return labels


def _build_caches():
    import tarfile
    if not TGZ_PATH.exists():
        raise FileNotFoundError(f'{TGZ_PATH} not found — check your Drive path.')

    # Pass 1: list inkml members (streaming; no disk writes).
    print('[subset] scanning archive index (one-time, a few minutes)…')
    train_names, val_names = [], []
    with tarfile.open(TGZ_PATH, 'r:gz') as tar:
        for m in tar:
            if not (m.isfile() and m.name.endswith('.inkml')):
                continue
            parts = m.name.split('/')
            if 'train' in parts:
                train_names.append(m.name)
            elif 'valid' in parts:
                val_names.append(m.name)
    print(f'[subset] archive: {len(train_names)} train / {len(val_names)} valid inkml')

    rng = random.Random(SEED)
    train_pick = set(rng.sample(train_names, min(N_TRAIN, len(train_names))))
    val_pick   = set(rng.sample(val_names,   min(N_VAL,   len(val_names))))

    # Pass 2: extract only the picked members.
    print(f'[subset] extracting {len(train_pick)} + {len(val_pick)} files…')
    with tarfile.open(TGZ_PATH, 'r:gz') as tar:
        for m in tar:
            if m.name in train_pick or m.name in val_pick:
                tar.extract(m, EXTRACT_DIR)

    train_paths = [EXTRACT_DIR / n for n in sorted(train_pick)]
    val_paths   = [EXTRACT_DIR / n for n in sorted(val_pick)]

    # Vocab from the train subset.
    print('[vocab] building from train subset…')
    labels = _collect_labels_from_paths(train_paths)
    tok2idx, idx2tok = build_vocab(labels)
    meta = {'max_points': MAX_POINTS, 'blank_idx': BLANK_IDX, 'vocab_size': len(idx2tok)}
    save_vocab(idx2tok, meta, VOCAB_CACHE)

    # Precompute features once (drops bad / all-OOV samples).
    print('[precompute] extracting features…')
    train_items = tl.precompute_items(train_paths, tok2idx)
    val_items   = tl.precompute_items(val_paths,   tok2idx)

    # Cache everything to Drive.
    with open(FEATURES_CACHE, 'wb') as f:
        pickle.dump({'train': train_items, 'val': val_items,
                     'vocab_size': len(idx2tok)}, f)
    with open(SUBSET_CACHE, 'w') as f:
        json.dump({'seed': SEED,
                   'train': [str(p) for p in train_paths],
                   'val':   [str(p) for p in val_paths]}, f)
    print('[cache] saved features + vocab + subset list to Drive.')
    return train_items, val_items, len(idx2tok)


if FEATURES_CACHE.exists():
    print('[cache] loading precomputed features from Drive…')
    with open(FEATURES_CACHE, 'rb') as f:
        blob = pickle.load(f)
    train_items, val_items, vocab_size = blob['train'], blob['val'], blob['vocab_size']
else:
    train_items, val_items, vocab_size = _build_caches()

print(f'\nusable: {len(train_items)} train / {len(val_items)} val  |  vocab_size = {vocab_size}')

## 7. DataLoaders (features already extracted → cheap collate)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tl.CachedDataset(train_items), batch_size=BATCH_SIZE,
    shuffle=True,  num_workers=2, collate_fn=_collate_fn,
)
val_loader = DataLoader(
    tl.CachedDataset(val_items), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, collate_fn=_collate_fn,
)
print('train batches:', len(train_loader), '| val batches:', len(val_loader))

## 8. Run the search (resumable)

The study lives in a SQLite file on Drive. `load_if_exists=True` reattaches to it,
and we only run the trials still owed toward `N_TRIALS`, so re-running this cell
after a disconnect continues the search instead of restarting it.

`safe_objective` turns an occasional CUDA OOM into a pruned trial rather than a
crash that ends the whole study.

In [ ]:
import optuna

def safe_objective(trial):
    try:
        return tl.objective(trial, train_loader, val_loader, vocab_size, device)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise optuna.TrialPruned('CUDA OOM — reduce BATCH_SIZE if this recurs.')

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(),
    load_if_exists=True,
)

_finished = (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)
done = sum(1 for t in study.trials if t.state in _finished)
remaining = max(0, N_TRIALS - done)
print(f'{done} trials finished; running {remaining} more (target {N_TRIALS}).')

if remaining:
    study.optimize(safe_objective, n_trials=remaining, gc_after_trial=True)
else:
    print('Target trial count already reached — nothing to do.')

## 9. Results

In [ ]:
print('Best val CER :', study.best_value)
print('Best params  :', study.best_params)

with open(BEST_PARAMS, 'w') as f:
    json.dump({'best_value': study.best_value,
               'best_params': study.best_params,
               'n_trials_finished': sum(1 for t in study.trials if t.state in _finished),
               'subset': {'n_train': len(train_items), 'n_val': len(val_items),
                          'seed': SEED},
               'fixed': {'batch_size': BATCH_SIZE, 'num_epochs': NUM_EPOCHS,
                         'dropout': tl.DROPOUT}}, f, indent=2)
print('saved →', BEST_PARAMS)

### Visualizations

In [ ]:
import optuna.visualization.matplotlib as ovm
import matplotlib.pyplot as plt

ovm.plot_optimization_history(study); plt.tight_layout(); plt.show()
ovm.plot_param_importances(study);   plt.tight_layout(); plt.show()
try:
    ovm.plot_contour(study, params=['learning_rate', 'warmup_ratio'])
    plt.tight_layout(); plt.show()
except Exception as e:
    print('contour skipped:', e)

## Notes & troubleshooting

- **OOM on the T4?** Set `BATCH_SIZE = 16` in cell 4 and re-run from cell 7. `gc_after_trial=True`
  and `safe_objective` already release memory between trials.
- **Disconnected?** Just re-run cells 1–8. The feature cache loads from Drive and the study
  resumes from the SQLite file — no re-extraction, same subset.
- **Fresh search?** Bump `STUDY_NAME` in cell 4 (new study + new cache files). Change `SEED`
  to draw a different random subset.
- **For the full run on `main`:** rebuild vocab from the *full* train split, drop the subset
  sampling, and revisit the M4 (`torch.load` `map_location`) and L5 (CTC length tensors on CPU)
  notes before training for real.